In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

/export/usuarios01/zhuldyz/Zhuldyz_files/DiffuSeq/.diffseq/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import json
from pathlib import Path

def load_json_or_jsonl(path):
    path = Path(path)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    if not text:
        return []

    # First try normal JSON
    try:
        data = json.loads(text)
        if isinstance(data, dict) and "data" in data:
            return data["data"]
        if isinstance(data, list):
            return data
        return [data]
    except json.JSONDecodeError:
        pass

    # Fallback: JSON Lines
    records = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            records.append(json.loads(line))
    return records


In [3]:
def flatten_recovery_records(records):
    rows = []

    for i, rec in enumerate(records):
        sequence_id = rec.get("sequence_id", f"seq_{i:06d}")
        source = rec.get("source", "")
        reference = rec.get("reference", "")

        for step in rec.get("recover_steps", []):
            rows.append({
                "sequence_id": sequence_id,
                "source": source,
                "reference": reference,
                "step_index": step.get("step_index"),
                "timestep": step.get("timestep"),
                "recover": step.get("recover", "")
            })

    return pd.DataFrame(rows)

In [4]:
records = load_json_or_jsonl("generation_outputs/diffuseq_qqp_h128_lr0.0001_t2000_sqrt_lossaware_seed102_qqp20260418-19:29:44/ema_0.9999_030000.pt.samples/seed123_step0.intermediate.json")
df = flatten_recovery_records(records)
df

,sequence_id,source,reference,step_index,timestep,recover
0,seq_000000,[CLS] astrology : i am a capricorn sun cap moo...,"[CLS] i ' m a triple capricorn ( sun, moon and...",0,1999,[CLS] astrology : i a cap cap please cap pleas...
1,seq_000000,[CLS] astrology : i am a capricorn sun cap moo...,"[CLS] i ' m a triple capricorn ( sun, moon and...",1,1998,[CLS] qualitylogy : to a please please cap don...
2,seq_000000,[CLS] astrology : i am a capricorn sun cap moo...,"[CLS] i ' m a triple capricorn ( sun, moon and...",2,1997,[CLS] curblogy : to a a please please please t...
3,seq_000000,[CLS] astrology : i am a capricorn sun cap moo...,"[CLS] i ' m a triple capricorn ( sun, moon and...",3,1996,[CLS] enoughlogy : using a a cap cap please tr...
4,seq_000000,[CLS] astrology : i am a capricorn sun cap moo...,"[CLS] i ' m a triple capricorn ( sun, moon and...",4,1995,[CLS] curblogy : a overcoming please please ca...
...,...,...,...,...,...,...
39995,seq_000019,[CLS] how is the new harry potter book ' harry...,[CLS] how bad is the new book by j. k rowling?...,1995,4,[CLS] what are the new one harry books potter ...
39996,seq_000019,[CLS] how is the new harry potter book ' harry...,[CLS] how bad is the new book by j. k rowling?...,1996,3,[CLS] what are the new one harry books potter ...
39997,seq_000019,[CLS] how is the new harry potter book ' harry...,[CLS] how bad is the new book by j. k rowling?...,1997,2,[CLS] what are the new one harry books potter ...
39998,seq_000019,[CLS] how is the new harry potter book ' harry...,[CLS] how bad is the new book by j. k rowling?...,1998,1,[CLS] what are the new one harry books potter ...


In [5]:
SPECIAL_TOKENS = [
    "[CLS]",
    "[SEP]",
    "[PAD]",
    "[UNK]",
    "[MASK]"
]

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    for tok in SPECIAL_TOKENS:
        text = text.replace(tok, " ")

    text = re.sub(r"\s+", " ", text).strip()
    return text

In [6]:
df["source_clean"] = df["source"].apply(clean_text)
df["reference_clean"] = df["reference"].apply(clean_text)
df["recover_clean"] = df["recover"].apply(clean_text)

df[["sequence_id", "step_index", "timestep", "recover_clean"]].head()

,sequence_id,step_index,timestep,recover_clean
0,seq_000000,0,1999,astrology : i a cap cap please cap please plea...
1,seq_000000,1,1998,qualitylogy : to a please please cap done plea...
2,seq_000000,2,1997,curblogy : to a a please please please truly p...
3,seq_000000,3,1996,enoughlogy : using a a cap cap please truly pl...
4,seq_000000,4,1995,curblogy : a overcoming please please cap₹ yet...


In [7]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed_unique_texts(df, text_columns, batch_size=128):
    unique_texts = pd.unique(
        pd.concat([df[col] for col in text_columns], ignore_index=True)
        .dropna()
        .astype(str)
    )

    unique_texts = [t for t in unique_texts if t.strip()]

    embeddings = model.encode(
        unique_texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    return dict(zip(unique_texts, embeddings))

In [8]:
embedding_lookup = embed_unique_texts(
    df,
    text_columns=["source_clean", "reference_clean", "recover_clean"],
    batch_size=128
)

def cosine_from_lookup(df, left_col, right_col, embedding_lookup):
    scores = np.full(len(df), np.nan)

    mask = (
        df[left_col].astype(str).str.strip().ne("") &
        df[right_col].astype(str).str.strip().ne("")
    )

    left_embeddings = np.vstack(
        df.loc[mask, left_col].map(embedding_lookup).to_numpy()
    )

    right_embeddings = np.vstack(
        df.loc[mask, right_col].map(embedding_lookup).to_numpy()
    )

    scores[mask.to_numpy()] = np.sum(left_embeddings * right_embeddings, axis=1)

    return scores

df["sim_recover_reference"] = cosine_from_lookup(
    df,
    "recover_clean",
    "reference_clean",
    embedding_lookup
)

df["sim_recover_source"] = cosine_from_lookup(
    df,
    "recover_clean",
    "source_clean",
    embedding_lookup
)

df["sim_source_reference"] = cosine_from_lookup(
    df,
    "source_clean",
    "reference_clean",
    embedding_lookup
)

df["sim_recover_reference_pct"] = (df["sim_recover_reference"] * 100).round(2)
df["sim_recover_source_pct"] = (df["sim_recover_source"] * 100).round(2)
df["sim_source_reference_pct"] = (df["sim_source_reference"] * 100).round(2)

df["avg_reference_source_similarity_pct"] = (
    df[["sim_recover_reference_pct", "sim_recover_source_pct"]]
    .mean(axis=1)
    .round(2)
)

df["gain_over_source_reference"] = (
    df["sim_recover_reference"] - df["sim_source_reference"]
).round(4)

df["gain_over_source_reference_pct"] = (
    df["gain_over_source_reference"] * 100
).round(2)

best_by_sequence = (
    df.sort_values(
        ["sequence_id", "sim_recover_reference"],
        ascending=[True, False]
    )
    .groupby("sequence_id", as_index=False)
    .first()
)

best_by_sequence[
    [
        "sequence_id",
        "step_index",
        "timestep",
        "sim_recover_reference_pct",
        "sim_recover_source_pct",
        "recover_clean"
    ]
].head()

df.to_parquet("scored_recovery_steps_baseline.parquet", index=False)
best_by_sequence.to_parquet("best_recovery_by_sequence_baseline.parquet", index=False)

Batches: 100%|██████████| 12/12 [00:00<00:00, 36.11it/s]


In [9]:

path = "/export/usuarios01/zhuldyz/Zhuldyz_files/DiffuSeq/best_recovery_by_sequence_baseline.parquet"

df = pd.read_parquet(path)
df.head()

,sequence_id,source,reference,step_index,timestep,recover,source_clean,reference_clean,recover_clean,sim_recover_reference,sim_recover_source,sim_source_reference,sim_recover_reference_pct,sim_recover_source_pct,sim_source_reference_pct,avg_reference_source_similarity_pct,gain_over_source_reference,gain_over_source_reference_pct
0,seq_000000,[CLS] astrology : i am a capricorn sun cap moo...,"[CLS] i ' m a triple capricorn ( sun, moon and...",1035,964,[CLS] astrology caporn about a cap cap and moo...,astrology : i am a capricorn sun cap moon and ...,"i ' m a triple capricorn ( sun, moon and ascen...",astrology caporn about a cap cap and moon?,0.628330,0.766627,0.837468,62.83,76.66,83.75,69.74,-0.2091,-20.91
1,seq_000001,[CLS] how can i be a good geologist? [SEP] [SEP],[CLS] what should i do to be a great geologist...,197,1802,[CLS] how can i become a good a geologist? [SEP],how can i be a good geologist?,what should i do to be a great geologist?,how can i become a good a geologist?,0.939280,0.982158,0.936959,93.93,98.22,93.70,96.08,0.0023,0.23
2,seq_000002,[CLS] how do i read and find my youtube commen...,[CLS] how can i see all my youtube comments? [...,67,1932,[CLS] how do i read my youtube comments? [SEP],how do i read and find my youtube comments?,how can i see all my youtube comments?,how do i read my youtube comments?,0.856612,0.953515,0.879711,85.66,95.35,87.97,90.50,-0.0231,-2.31
3,seq_000003,[CLS] what can make physics easy to learn? [SE...,[CLS] how can you make physics easy to learn? ...,46,1953,[CLS] what can make physics easy to learn? [SEP],what can make physics easy to learn?,how can you make physics easy to learn?,what can make physics easy to learn?,0.974253,1.000000,0.974253,97.43,100.00,97.43,98.72,0.0000,0.00
4,seq_000004,[CLS] what was your first sexual experience li...,[CLS] what was your first sexual experience? [...,58,1941,[CLS] what was your first sexual? experience,what was your first sexual experience like?,what was your first sexual experience?,what was your first sexual? experience,0.987813,0.954799,0.971795,98.78,95.48,97.18,97.13,0.0160,1.60
